# Homework: worked versions

Task 1 is your own question, so there is no single answer. Here is one
worked example, with the reasoning, to compare your choices against. Task 2
has a right direction, if not one right chart.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
os.makedirs("output", exist_ok=True)

plays = pd.read_csv("data/clean/plays.csv")
plays["year"] = plays["played_at"].str[:4].astype(int)
print(len(plays), "plays")

## Task 1: one chart that answers one question

**My question:** which artists got the most minutes in 2025?

In [ ]:
def top_artists_by_minutes(plays, year, n=8):
    """Return the n artists with the most minutes in a year, biggest first."""
    in_year = plays[plays["year"] == year]
    minutes = in_year.groupby("artist_name")["minutes_played"].sum()
    return minutes.sort_values(ascending=False).head(n)


top = top_artists_by_minutes(plays, 2025)
gap = top.iloc[0] - top.iloc[1]
print(top.round(1))
print(f"gap between the top two: {gap:.1f} minutes")

assert list(top.index[:2]) == ["DJ Kompas", "Glass Tram"]
assert round(gap, 1) == 25.4

Before writing a title, ask how sure it is. 25.4 minutes is five or six
plays. Is that a clear win?

One way to find out: the messy file is the version you would really have
had. 94 of its plays have no minutes. Take those out of the clean file and
see what the ranking would have looked like.

In [ ]:
messy = pd.read_csv("data/messy/plays_messy.csv", keep_default_na=False)
no_minutes = messy.loc[messy["minutes played "].str.strip() == "", "play_id"].unique()

known = plays[~plays["play_id"].isin(no_minutes)]
top_known = top_artists_by_minutes(known, 2025)
gap_known = top_known.iloc[0] - top_known.iloc[1]

in_2025 = plays[(plays["year"] == 2025) & plays["play_id"].isin(no_minutes)]
lost = in_2025["artist_name"].value_counts().reindex(top.index[:2])

print(f"{len(no_minutes)} plays with no minutes in the messy file")
print(f"gap with only the minutes the messy file knew: {gap_known:.1f}")
print(lost)

assert len(no_minutes) == 94
assert round(gap_known, 1) == 3.5
assert list(lost) == [6, 2]

With the file as you would have received it, the gap was 3.5 minutes, and
DJ Kompas had 6 plays with no length against Glass Tram's 2. Any one of
them could have been longer than the gap. With the clean file, DJ Kompas
wins by 25.4. So the title can say DJ Kompas came first, and should not
make it sound like a landslide.

`.reindex(top.index[:2])` puts the counts in the same order as the top two,
so the comparison lines up.

In [ ]:
ranked = top.sort_values()                      # barh draws from the bottom up
colours = ["#4f6ddb" if name in top.index[:2] else "#b8bec9" for name in ranked.index]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(ranked.index, ranked.values, color=colours)
ax.set_title(f"DJ Kompas edged out Glass Tram in 2025, by {gap:.1f} minutes")
ax.set_xlabel("minutes played in 2025")
ax.bar_label(bars, fmt="{:,.1f}", padding=3)
ax.set_xlim(0, top.iloc[0] * 1.15)
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/top_artists_2025.png", dpi=150, bbox_inches="tight")
plt.show()

The choices:

* horizontal bars, because artist names are long; still bars, because
  artists are categories
* sorted ascending before `barh`, so the biggest ends up at the top
* the top eight, not all 38: the question is about the top
* the two that the title is about in colour, the rest grey
* "edged out", not "beat": the title says what the data supports

**What I would say about it in a meeting:** "Two artists dominated my 2025,
DJ Kompas and Glass Tram, with over 540 minutes each, and the next one is
140 minutes behind."

**What this chart does not show:** how often each artist was played (this
is minutes, not plays), anything about 2024, and how fragile the order of
the top two is. With the 94 missing lengths of the messy file, the gap was
3.5 minutes, smaller than one song.

In [ ]:
assert top.iloc[1] > 540
assert round(top.iloc[1] - top.iloc[2]) == 142

---

## Task 2: a makeover

The chart you were given:

In [ ]:
shares = plays.pivot_table(index="device", columns="year", values="play_id", aggfunc="count")
shares = shares / shares.sum() * 100

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.5))
left.pie(shares[2024], labels=shares.index, startangle=90, shadow=True)
right.pie(shares[2025], labels=shares.index, startangle=0, shadow=True)
left.set_title("2024")
right.set_title("2025")
fig.suptitle("Devices")
plt.show()

**What is wrong with it.** Two pies to compare means comparing angles that
start in different places (the second pie is turned a quarter), on two
separate circles. The shadows add nothing. There are no numbers, and the
title is a topic. You cannot tell from it whether the phone went up or
down.

**The question:** is the phone losing ground to other devices?

That is a part-of-a-whole question across two years. The numbers are the
shares, and the thing to show is how each share moved. One good shape for
"before and after, per category" is a **dumbbell**: one row per device, a
grey dot for 2024, a coloured dot for 2025, and a line between them.

In [ ]:
shares["change"] = shares[2025] - shares[2024]
shares = shares.sort_values(2025)
shares.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hlines(shares.index, shares[2024], shares[2025], color="#b8bec9", linewidth=3)
ax.scatter(shares[2024], shares.index, color="#b8bec9", s=80, label="2024", zorder=3)
ax.scatter(shares[2025], shares.index, color="#0072b2", s=80, label="2025", zorder=3)

for device, row in shares.iterrows():
    ax.annotate(f"{row[2025]:.1f}%", (row[2025], device), xytext=(0, 9),
                textcoords="offset points", ha="center", color="#0072b2")

phone_2024, phone_2025 = shares.loc["phone", 2024], shares.loc["phone", 2025]
others_gained = (shares.drop("phone")["change"] > 0).all()

ax.set_title(f"The phone's share fell from {phone_2024:.1f}% to {phone_2025:.1f}%; "
             "every other device gained")
ax.set_xlabel("share of the year's plays (%)")
ax.set_xlim(0, 55)
ax.margins(y=0.15)
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.savefig("output/device_share_makeover.png", dpi=150, bbox_inches="tight")
plt.show()

assert (round(phone_2024, 1), round(phone_2025, 1)) == (49.9, 45.1)
assert others_gained

`ax.hlines(y, start, end)` draws one horizontal line per row, here from the
2024 share to the 2025 share. `zorder=3` puts the dots on top of the lines, and `ax.margins(y=0.15)` leaves
room above the top row for its label.
`iterrows()` walks a DataFrame one row at a time, giving the index and the
row, which is handy for one label per row.

A grouped bar chart, 2024 and 2025 side by side per device, would also be
a fine answer. What matters is that both years share one axis starting at
zero, and the reader compares lengths or positions, not angles.

**What this chart does not show:** whether anything was played **less**.
Shares only say how the year was split. Look at the counts:

In [ ]:
counts = plays.pivot_table(index="device", columns="year", values="play_id", aggfunc="count")
counts

In [ ]:
assert (counts.loc["phone", 2024], counts.loc["phone", 2025]) == (560, 479)
assert (counts.loc["laptop", 2024], counts.loc["laptop", 2025]) == (263, 261)

Phone plays fell from 560 to 479. And the laptop's share went up, from 23.4%
to 24.6%, while its plays went **down**, from 263 to 261. Both are true:
listening as a whole fell by more than the laptop did. "Every other device
gained" is true of shares and false of plays, so the title says "share".